# Generative AI 019 — Tool Calling

Binding, calling, execution. Only *calling* needs a real model to decide; a
scripted model stands in for that step and is labelled. Everything else is
real, and needs **no API key**.

| Part | What we check |
|---|---|
| A | binding attaches a **259-character** schema; the model's reply is a request |
| B | `invoke(args)` gives a value, `invoke(tool_call)` gives a `ToolMessage` |
| C | `InjectedToolArg` removes the rate from the model's schema |
| D | the source's execution loop breaks if the calls arrive in the other order |

Needs `langchain-core`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — Binding and calling

In [ ]:
import json
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def multiply(a: int, b: int) -> int:
    """Given two numbers a and b, this tool returns their product"""
    return a * b

class ScriptedModel(GenericFakeChatModel):
    """Stands in for a tool-calling model: replays pre-written AIMessages.
    LangChain's stubs raise NotImplementedError on bind_tools, so accept it."""
    def bind_tools(self, tools, **kwargs):
        self._bound = [convert_to_openai_tool(t) for t in tools]
        return self

# ---- BINDING: what gets attached to every request ---------------------------
model = ScriptedModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "multiply", "args": {"a": 3, "b": 10},
                                       "id": "call_1"}]),
    AIMessage(content="The product of 3 and 10 is 30."),
])).bind_tools([multiply])
print(json.dumps(model._bound))       # 259 characters of schema, on every call

# ---- CALLING: the model's reply is a request, not a result (SCRIPTED) -------
messages = [HumanMessage("Can you multiply 3 with 10?")]
ai = model.invoke(messages)
print(repr(ai.content), ai.tool_calls)
# '' [{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_1', 'type': 'tool_call'}]
#
# Empty content. Nothing has been multiplied. The model ASKED; your code decides.

In [ ]:
assert len(json.dumps(model._bound)) == 259
assert ai.content == "" and ai.tool_calls[0]["name"] == "multiply"
print("a request, not a result")

## Part B — Execution

In [ ]:
# ---- EXECUTION: two ways to invoke, two different results -------------------
call = ai.tool_calls[0]
print(multiply.invoke(call["args"]))    # 30            - just the value
msg = multiply.invoke(call)             # ToolMessage   - the value, labelled
print(type(msg).__name__, repr(msg.content), msg.tool_call_id)
# ToolMessage '30' call_1
#
# The tool_call_id matters: one reply can ask for several calls, and each
# answer has to be matched to its question.

messages.append(ai)
messages.append(msg)
messages.append(model.invoke(messages))      # the model now sees the result
for m in messages:
    print(f"{type(m).__name__:<12}", m.content or m.tool_calls)
# HumanMessage Can you multiply 3 with 10?
# AIMessage    [{'name': 'multiply', ...}]
# ToolMessage  30
# AIMessage    The product of 3 and 10 is 30.

In [ ]:
assert multiply.invoke(call["args"]) == 30
assert type(msg).__name__ == "ToolMessage" and msg.tool_call_id == "call_1"
assert messages[-1].content == "The product of 3 and 10 is 30."

## Part C — InjectedToolArg

The API response is **canned** — there is no network here. The point is what
the model is and is not asked for.

In [ ]:
from typing import Annotated
from langchain_core.tools import InjectedToolArg

LIVE = '{"base_code": "USD", "target_code": "INR", "conversion_rate": 83.12}'   # CANNED

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> str:
    """Fetch the conversion factor between a base and a target currency."""
    return LIVE          # a real version calls an exchange-rate API

@tool
def convert(base_currency_value: int,
            conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """Given a conversion rate, calculate the target currency value."""
    return base_currency_value * conversion_rate

print(list(convert_to_openai_tool(convert)["function"]["parameters"]["properties"]))
# ['base_currency_value']      <- the model is never asked for the rate

tc = {"name": "convert", "args": {"base_currency_value": 10}, "id": "c2", "type": "tool_call"}
try:
    convert.invoke(tc)
except Exception as e:
    print(type(e).__name__)      # ValidationError - forgetting to inject is LOUD

tc["args"]["conversion_rate"] = json.loads(
    get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})
)["conversion_rate"]
print(convert.invoke(tc).content)            # 831.2
#
# Without InjectedToolArg the schema asks the model for the rate in the SAME
# reply as the call that fetches it - so it answers from stale training data
# (the source reports 74.53). With it, the rate can only come from your code.

In [ ]:
assert list(convert_to_openai_tool(convert)["function"]["parameters"]["properties"]) \
    == ["base_currency_value"]
assert abs(float(convert.invoke(tc).content) - 831.2) < 1e-9

## Part D — The order bug

In [ ]:
rate_call = {"name": "get_conversion_factor", "id": "c1", "type": "tool_call",
             "args": {"base_currency": "USD", "target_currency": "INR"}}
conv_call = {"name": "convert", "id": "c2", "type": "tool_call",
             "args": {"base_currency_value": 10}}

def source_loop(tool_calls):
    """The loop exactly as the source writes it."""
    out = []
    for tc in tool_calls:
        tc = dict(tc, args=dict(tc["args"]))
        if tc["name"] == "get_conversion_factor":
            m1 = get_conversion_factor.invoke(tc)
            conversion_rate = json.loads(m1.content)["conversion_rate"]
            out.append(m1)
        if tc["name"] == "convert":
            tc["args"]["conversion_rate"] = conversion_rate
            out.append(convert.invoke(tc))
    return out

print(source_loop([rate_call, conv_call])[-1].content)      # 831.2
try:
    source_loop([conv_call, rate_call])
except UnboundLocalError as e:
    print("UnboundLocalError:", e)
# The loop works only if the model LISTS the rate call first. Nothing
# guarantees that order. Run what a result depends on first:

def safe_loop(tool_calls):
    by_name = {tc["name"]: dict(tc, args=dict(tc["args"])) for tc in tool_calls}
    rate = json.loads(get_conversion_factor.invoke(
        by_name["get_conversion_factor"]).content)["conversion_rate"]
    by_name["convert"]["args"]["conversion_rate"] = rate
    return convert.invoke(by_name["convert"])

print(safe_loop([conv_call, rate_call]).content)             # 831.2, either order
# And notice who decided that order: the programmer. That is why this is
# tool calling, not yet an agent.

## What to take away

- Binding attaches a schema to **every** request; a tool call is a **request**.
- `invoke(tool_call)` returns a `ToolMessage` whose id matches its question.
- `InjectedToolArg` hides an argument the model must not invent — and
  forgetting to inject fails loudly.
- Run what a result depends on **first**. The programmer deciding the order is
  why this is not yet an agent.

## Exercises

1. Bind three tools and measure the schema that is sent. What does each extra
   tool cost per call?
2. Make the scripted model return two tool calls for the *same* tool with
   different arguments. Does the tool_call_id still keep the answers apart?
3. Mark a second argument with `InjectedToolArg` — a user id, say. Why is that a
   good idea for anything touching user data?
4. Write `safe_loop` for three dependent tools, where the third needs both
   earlier results.